# SAVER-VLA on Colab

Runs the `saver_vla` package with LIBERO and SmolVLA on a Colab GPU.
LeRobot is installed **unmodified** at v0.6.1: no patch scripts, no `SAVER_*` environment variables.

Runtime: **GPU** (Runtime → Change runtime type). Steps 1–3 take about 10 minutes once per session.

## 1. Install LeRobot v0.6.1 with LIBERO and SmolVLA

In [ ]:
%cd /content
!git clone -q https://github.com/huggingface/lerobot.git
%cd /content/lerobot
!git checkout -q v0.6.1 && git rev-parse HEAD
!apt-get update -qq && apt-get install -y -qq cmake build-essential libegl1-mesa-dev libgl1-mesa-dev

import os
os.environ["PATH"] = "/usr/bin:" + os.environ["PATH"]
os.environ["CMAKE_POLICY_VERSION_MINIMUM"] = "3.5"
!pip install -q --no-cache-dir hf-egl-probe==1.0.2 egl_probe==1.0.2
!pip install -q -e ".[libero,smolvla]"
!pip install -q "mujoco==3.3.2"
!pip uninstall -y -q torchao

In [ ]:
import os, subprocess, sys
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# LIBERO asks a question on first import; answer it once.
print(subprocess.run([sys.executable, "-c", "import libero.libero"], input="N\n",
                     text=True, capture_output=True, timeout=120).returncode)

import torch
assert torch.cuda.is_available(), "Select a GPU runtime"
print("GPU:", torch.cuda.get_device_name(0))

## 2. Install `saver-vla`

Either upload `saver-vla.zip` to `/content` (left sidebar → Files), or clone your GitHub repo.

In [ ]:
%cd /content
# !git clone https://github.com/mehdiomid/saver-vla.git

!pip install -q -e "/content/saver-vla[llm,dev]"
%cd /content/saver-vla

## 3. Fast checks (no simulator): the unit tests and the toy world

In [ ]:
!python -m pytest -q
!saver run configs/toy_demo.yaml --quiet --out /content/runs/toy_demo

## 4. Validation: replay the original Phase 6.5.1 decisions

Upload the notebook-era log `phase651_geometry_gpt55_5.5cm/agent_events.jsonl` to `/content`.
The replay agent re-issues GPT-5.5's eight recorded decisions in order, with no API calls.
If the new runner reproduces the original episode, the decision steps should match the
original run (80, 127, 200, 295, 515, 599, 672, 740) and the second grasp trial should be verified.

In [ ]:
!saver run configs/phase651.yaml --quiet \
    --set recovery.agent=replay \
    --set recovery.replay_log=/content/agent_events.jsonl \
    --out /content/runs/phase651_replay

In [ ]:
from saver_vla.events import read_events
for e in read_events("/content/runs/phase651_replay/events.jsonl"):
    if e["kind"] == "DECISION":
        print(e["step"], e["executed_skill"], e["variant"])
    elif e["kind"] in ("SKILL_OUTCOME", "POLICY_TRIAL_OUTCOME"):
        print("   ", e["kind"], e.get("phase") or e["status"]["phase"])
print(open("/content/runs/phase651_replay/result.json").read())

## 5. Live GPT-5.5 run

Store your key in Colab Secrets as `OPENAI_API_KEY` (key icon in the left sidebar).

In [ ]:
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
!saver run configs/phase651.yaml --out /content/runs/phase651_live

## 6. Reliability sweep

Lists the episodes first, then runs them. Finished episodes are skipped on re-run, so
an interrupted session can continue. Save `/content/runs` to Google Drive to keep results.

> Earlier exploratory sweep, kept for reference. **For the paper experiments, use section 9.**

In [ ]:
!saver sweep configs/sweep_reliability.yaml --dry-run | head -20

In [ ]:
!saver sweep configs/sweep_reliability.yaml --set output_dir=/content/runs
!cat /content/runs/reliability_v1/report.md

## 7. Hard scenarios

First record three episodes and watch them, to see what the randomized faults look like. Then run the full sweep. Results are saved after every episode, so if Colab disconnects, re-run the same cell to continue.

> Earlier exploratory sweep, kept for reference. **For the paper experiments, use section 9.**

In [ ]:
!saver run configs/phase651.yaml --quiet --set env.record_video=true --set faults.transport=forced_separation \
    --set "faults.displacement_m=[0.06, 0.22]" --set faults.direction=random --set seed=2000 --set episode_index=3 \
    --out /content/runs/watch_S1
from IPython.display import Video
Video("/content/runs/watch_S1/episode.mp4", embed=True, width=480)

In [ ]:
# Keep results across sessions
from google.colab import drive
drive.mount("/content/drive")
!saver sweep configs/sweep_hard.yaml --set output_dir=/content/drive/MyDrive/saver_runs
!cat /content/drive/MyDrive/saver_runs/hard_v1/report.md

## 8. Other objects (generalization)

> Earlier exploratory sweep, kept for reference. **For the paper experiments, use section 9.**

In [ ]:
!saver sweep configs/sweep_generalization.yaml --set output_dir=/content/drive/MyDrive/saver_runs
!cat /content/drive/MyDrive/saver_runs/generalization_v1/report.md

## 9. Paper experiments

Run these in order. Each answers a reviewer question and is useful on its own if you stop early.
Results go to Google Drive after every episode: if Colab disconnects, re-run the same cell and
finished episodes are skipped. Each sweep first makes one test call to GPT-5.5 and stops at once
if the API is unavailable (for example, no credits).

| Step | Experiment | Episodes (GPT-5.5) | Time |
|---|---|---|---|
| 9.1 | E1 main reliability | 128 (32) | ~4 h |
| 9.2 | E2 hard scenarios: LLM vs. script | 80 (40) | ~2.5 h |
| 9.3 | E3 ablations | 48 (40) | ~1.5 h |
| 9.4 | Optional: policy rescue, generalization | 105 (35) | ~3 h |

Before starting: add OpenAI credits and set a usage limit. After each step, download
`report.md` and `comparisons.md` from the run folder.

In [ ]:
# Mount Drive and set the API key (run once per session)
import os
from google.colab import drive, userdata
drive.mount("/content/drive")
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
RUNS = "/content/drive/MyDrive/saver_runs"

### 9.1 E1: main reliability (essential)
4 controllers (SmolVLA alone, queue reset, scripted, GPT-5.5) x 2 randomized fault families x 16 episodes.

In [ ]:
!saver sweep /content/saver-vla/configs/paper_E1_main.yaml --set output_dir=$RUNS
!saver report  $RUNS/paper_E1_main
!saver compare $RUNS/paper_E1_main

### 9.2 E2: hard scenarios (essential)
GPT-5.5 versus the fixed script on faults that need adaptation, with paired episodes (identical faults). Replaces `sweep_hard_llm.yaml`.

In [ ]:
!saver sweep /content/saver-vla/configs/paper_E2_hard.yaml --set output_dir=$RUNS
!saver report  $RUNS/paper_E2_hard
!saver compare $RUNS/paper_E2_hard

### 9.3 E3: ablations (strongly recommended)
Removes one component at a time: verified grasp trials, capability information, geometry skills; plus fault disclosed and a scripted reference.

In [ ]:
!saver sweep /content/saver-vla/configs/paper_E3_ablations.yaml --set output_dir=$RUNS
!saver report $RUNS/paper_E3_ablations

### 9.4 Optional
Re-screen the failing initial states (free), then test whether SAVER-VLA rescues SmolVLA's own failures, and generalization to other objects.

In [ ]:
!saver sweep /content/saver-vla/configs/nominal_rescreen.yaml --set output_dir=$RUNS
!saver report $RUNS/nominal_rescreen

In [ ]:
!saver sweep /content/saver-vla/configs/paper_E5_policy_rescue.yaml --set output_dir=$RUNS
!saver report $RUNS/paper_E5_policy_rescue

In [ ]:
!saver sweep /content/saver-vla/configs/sweep_generalization.yaml --set output_dir=$RUNS
!saver report  $RUNS/generalization_v1
!saver compare $RUNS/generalization_v1

### Download all reports
Creates one small zip with every report, comparison and LaTeX table (not the large event logs).

In [ ]:
!cd $RUNS && zip -q -r /content/saver_reports.zip */report.md */comparisons.md */summary.csv */summary.json */table.tex 2>/dev/null; ls -la /content/saver_reports.zip
from google.colab import files
files.download("/content/saver_reports.zip")